# IEEE118 EMT component timing comparison

Fresh detailed profiles use Δt = 50 µs and T_sim = 30 s. SSN, Aggregate, the original Grouped passive huge SSN, and the new Grouped passive huge SSN — shared maps each have five sequential wall-time runs; Conventional has one completed run as requested. Component-family and stage bars use the representative median-wall-time profile for each configuration, while wall-to-wall values use the measured mean (or the single conventional run). The new shared-map profile is kept separate from the earlier grouped baseline.

In [ ]:
from pathlib import Path
import csv
import lzma
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("examples/Notebooks/Benchmarks/data/verified_rerun")
if not DATA.exists():
    DATA = Path("data/verified_rerun")
CONFIGS = [
    "conventional",
    "ssn",
    "aggregate",
    "grouped_passive_huge_ssn",
    "grouped_shared_maps",
]
DISPLAY = {
    "conventional": "Conventional (n=1)",
    "ssn": "SSN (n=5)",
    "aggregate": "Aggregate (n=5)",
    "grouped_passive_huge_ssn": "Grouped passive huge SSN — previous (n=5)",
    "grouped_shared_maps": "Grouped passive huge SSN — shared maps (n=5)",
}
DISPLAY_ORDER = [DISPLAY[c] for c in CONFIGS]
COLORS = ["#4472c4", "#ed7d31", "#59a14f", "#b279a2", "#e15759"]


def load_tasks(configuration):
    rows = []
    with (DATA / f"{configuration}_TaskTimes.csv").open() as f:
        for name, avg_ns, min_ns, max_ns, calls in csv.reader(f):
            stage = name.rsplit(".", 1)[-1] if "." in name else "other"
            component = name.rsplit(".", 1)[0] if "." in name else name
            if stage in {"MnaPreStep", "MnaPostStep", "Solve"}:
                rows.append(
                    {
                        "configuration": configuration,
                        "component": component,
                        "stage": stage,
                        "avg_us": int(avg_ns) / 1000,
                        "calls": int(calls),
                        "cumulative_s": int(avg_ns) * int(calls) / 1e9,
                    }
                )
    return pd.DataFrame(rows)


tasks = pd.concat([load_tasks(c) for c in CONFIGS], ignore_index=True)


def family(name):
    if name.startswith("Gen"):
        return "Generators"
    if name.startswith("Trafo"):
        return "Transformers"
    if name.startswith("LinearNetworkGroup"):
        return "Grouped passive huge SSN"
    if name.startswith("LinearNetwork"):
        return "Linear aggregate"
    if name.startswith("Line"):
        return "PiLines"
    if name.startswith(("ImpedanceLoad", "Load", "Shunt")):
        return "Loads + shunts"
    return "Other"


tasks["family"] = tasks["component"].map(family)
component_tasks = tasks[tasks.stage.isin(["MnaPreStep", "MnaPostStep"])]
family_totals = (
    component_tasks.groupby(["configuration", "family", "stage"])["cumulative_s"]
    .sum()
    .unstack("stage", fill_value=0)
    .reset_index()
)
for stage_name in ["MnaPreStep", "MnaPostStep"]:
    if stage_name not in family_totals:
        family_totals[stage_name] = 0.0
family_totals["total_s"] = family_totals["MnaPreStep"] + family_totals["MnaPostStep"]
families = [
    "PiLines",
    "Generators",
    "Transformers",
    "Loads + shunts",
    "Linear aggregate",
    "Grouped passive huge SSN",
]
family_totals = family_totals[family_totals.family.isin(families)]
family_totals

In [ ]:
# Figure 1: cumulative pre/post component time by family, from the verified profiles.
wide = (
    family_totals.pivot(index="family", columns="configuration", values="total_s")
    .reindex(families)
    .fillna(0)
)
wide = wide.rename(columns=DISPLAY).reindex(columns=DISPLAY_ORDER)
ax = wide.plot.bar(figsize=(14, 5.5), color=COLORS, width=0.82)
ax.set_title(
    "Component-family timing: conventional, SSN, aggregate, grouped baseline, and shared-map grouped SSN\nIEEE118 EMT | Δt = 50 µs | T_sim = 30 s"
)
ax.set_ylabel("pre-step + post-step cumulative seconds")
ax.set_xlabel("component type")
ax.grid(axis="y", alpha=0.25)
ax.legend(title="configuration", fontsize=8)
ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: profiled simulation stages and measured wall-to-wall time.
n_steps = 599999
stage = (
    tasks.groupby(["configuration", "stage"])["cumulative_s"]
    .sum()
    .unstack(fill_value=0)
)
wall_mean = {
    "conventional": 248.06,
    "ssn": 81.488,
    "aggregate": 69.246,
    "grouped_passive_huge_ssn": 85.462,
    "grouped_shared_maps": 71.624,
}
stage_plot = pd.DataFrame(
    {
        "Pre-step": stage["MnaPreStep"],
        "Post-step": stage["MnaPostStep"],
        "Solve": stage["Solve"],
        "Wall-to-wall": pd.Series(wall_mean),
    }
)
stage_plot = stage_plot.rename(index=DISPLAY).reindex(DISPLAY_ORDER)
ax = stage_plot.plot.bar(
    figsize=(14, 5.5), color=["#70ad47", "#ffc000", "#a5a5a5", "#4472c4"], width=0.82
)
ax.set_title(
    "Profiled stage totals and wall time\nIEEE118 EMT | Δt = 50 µs | T_sim = 30 s"
)
ax.set_ylabel("seconds")
ax.set_xlabel("configuration")
ax.grid(axis="y", alpha=0.25)
ax.legend(ncol=4, loc="upper left")
ax.set_xticklabels(
    [
        "Conventional (1 run)",
        "SSN (5-run mean)",
        "Aggregate (5-run mean)",
        "Grouped passive huge SSN — previous (5-run mean)",
        "Grouped passive huge SSN — shared maps (5-run mean)",
    ],
    rotation=12,
    ha="right",
)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f", padding=2, fontsize=8)
plt.tight_layout()
plt.show()
display(stage_plot.round(3))

In [ ]:
# Figure 3: compare pre-step with pre-step and post-step with post-step by family.
pre = (
    family_totals.pivot(index="family", columns="configuration", values="MnaPreStep")
    .reindex(families)
    .fillna(0)
)
post = (
    family_totals.pivot(index="family", columns="configuration", values="MnaPostStep")
    .reindex(families)
    .fillna(0)
)
pre = pre.rename(columns=DISPLAY).reindex(columns=DISPLAY_ORDER)
post = post.rename(columns=DISPLAY).reindex(columns=DISPLAY_ORDER)
fig, axes = plt.subplots(1, 2, figsize=(18, 5.5), sharey=False)
pre.plot.bar(ax=axes[0], color=COLORS, width=0.82)
post.plot.bar(ax=axes[1], color=COLORS, width=0.82)
for ax, title in zip(
    axes, ["Pre-step by component family", "Post-step by component family"]
):
    ax.set_title(title)
    ax.set_ylabel("cumulative seconds")
    ax.set_xlabel("component type")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(title="configuration", fontsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
fig.suptitle(
    "Component-family timing by stage\nIEEE118 EMT | Δt = 50 µs | T_sim = 30 s"
)
fig.tight_layout()
plt.show()

In [ ]:
def load_steps(configuration):
    values = []
    step_path = DATA / f"{configuration}_StepTimes.log"
    if step_path.exists():
        source = step_path.open()
    else:
        source = lzma.open(DATA / f"{configuration}_StepTimes.log.xz", "rt")
    with source as f:
        for line in f:
            value = line.strip()
            if value and value != "step_time":
                values.append(float(value) * 1000.0)
    return np.asarray(values)


assert set(tasks.stage.unique()) == {"MnaPreStep", "MnaPostStep", "Solve"}
assert all(load_steps(c).size == n_steps for c in CONFIGS)
assert wall_mean == {
    "conventional": 248.06,
    "ssn": 81.488,
    "aggregate": 69.246,
    "grouped_passive_huge_ssn": 85.462,
    "grouped_shared_maps": 71.624,
}
print(f"All five configurations have {n_steps:,} measured steps.")
print(
    "Wall time: conventional n=1; SSN, aggregate, previous grouped, and shared-map grouped runs n=5 each."
)

In [ ]:
# Per-step duration distributions from the same representative verified profiles.
step_labels = DISPLAY_ORDER
step_arrays = [load_steps(c) for c in CONFIGS]
fig, ax = plt.subplots(figsize=(13, 4.8))
ax.boxplot(step_arrays, tick_labels=step_labels, showfliers=False, showmeans=True)
ax.set_title(
    "Simulation-step duration distribution (outliers hidden)\nIEEE118 EMT | Δt = 50 µs | T_sim = 30 s"
)
ax.set_ylabel("milliseconds per step")
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
pd.DataFrame(
    {
        "configuration": step_labels,
        "mean_ms": [x.mean() for x in step_arrays],
        "median_ms": [np.median(x) for x in step_arrays],
        "p95_ms": [np.percentile(x, 95) for x in step_arrays],
        "max_ms": [x.max() for x in step_arrays],
    }
).round(4)

In [ ]:
# Figure 5: detailed internal solver phases for the new shared-map 30-second run.
# These profiler counters include nested timers (for example, DenseLU corrections
# are inside numeric_solve, and RHS subphases are inside rhs_assembly); do not sum them.
phase = pd.read_csv(DATA / "grouped_shared_maps_SolverPhaseTimes.csv")
phase = phase[(phase["total_s"] > 0) & (~phase["phase"].str.endswith("_calls"))].copy()
phase = phase.sort_values("total_s")
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(phase["phase"], phase["total_s"], color="#e15759")
ax.set_title(
    "Grouped passive SSN — shared maps: detailed solver phase profile\n"
    "IEEE118 EMT | Δt = 50 µs | T_sim = 30 s | median of 5 runs"
)
ax.set_xlabel("cumulative profiled seconds (nested counters overlap)")
ax.grid(axis="x", alpha=0.25)
for y, value in enumerate(phase["total_s"]):
    ax.text(value, y, f" {value:.2f} s", va="center", fontsize=8)
plt.tight_layout()
plt.show()
display(
    phase[["phase", "calls", "total_s", "avg_us"]]
    .sort_values("total_s", ascending=False)
    .round({"total_s": 3, "avg_us": 3})
    .reset_index(drop=True)
)